Este notebook utiliza o MediaPipe para a detecção e visualização de pontos de referência (landmarks) do corpo humano (face, pose e mãos) via **frames da webcam**.

Há uma função que detecta todos os landmarks a partir de um número pré determinado de frames e outra função que capta os frames em loop mas detecta apenas os landmarks da mão. É apenas um algoritmo de teste, pois usar o Colab para detectar imagens de webcam não é o melhor caminho. O algoritmo detect_smile_basic.py é uma referência melhor.




In [ ]:
# instalar biblioteca
!pip install mediapipe==0.10.14

### Coleta imagens da webcan a partir de um número pré-determinado de frames

In [ ]:
from google.colab import output
import base64
from io import BytesIO
from PIL import Image
import numpy as np

def get_frame():
    js = """
    async function captureFrame() {
      const stream = await navigator.mediaDevices.getUserMedia({ video: true });
      const video = document.createElement('video');
      video.srcObject = stream;
      await video.play();

      await new Promise(resolve => setTimeout(resolve, 200));

      const canvas = document.createElement('canvas');
      canvas.width = video.videoWidth;
      canvas.height = video.videoHeight;
      canvas.getContext('2d').drawImage(video, 0, 0);

      stream.getVideoTracks()[0].stop();
      return canvas.toDataURL('image/jpeg', 0.9);
    }
    captureFrame();
    """

    data = output.eval_js(js)
    img_bytes = base64.b64decode(data.split(',')[1])
    img = np.array(Image.open(BytesIO(img_bytes)))
    return img


In [ ]:
import mediapipe as mp
import cv2
import matplotlib.pyplot as plt

mp_holistic = mp.solutions.holistic
mp_draw = mp.solutions.drawing_utils

holistic = mp_holistic.Holistic(
    model_complexity=1,
    smooth_landmarks=True,
    refine_face_landmarks=True
)

def process_and_show():
    frame = get_frame()

    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = holistic.process(rgb)

    output_frame = frame.copy()

    if results.pose_landmarks:
        mp_draw.draw_landmarks(
            output_frame,
            results.pose_landmarks,
            mp_holistic.POSE_CONNECTIONS
        )

    if results.face_landmarks:
        mp_draw.draw_landmarks(
            output_frame,
            results.face_landmarks,
            mp_holistic.FACEMESH_TESSELATION
        )

    if results.left_hand_landmarks:
        mp_draw.draw_landmarks(
            output_frame,
            results.left_hand_landmarks,
            mp_holistic.HAND_CONNECTIONS
        )

    if results.right_hand_landmarks:
        mp_draw.draw_landmarks(
            output_frame,
            results.right_hand_landmarks,
            mp_holistic.HAND_CONNECTIONS
        )

    plt.figure(figsize=(6,6))
    plt.imshow(output_frame)
    plt.axis('off')
    plt.show()


In [ ]:
for i in range(10):   # número de frames que quer processar
    process_and_show()

### Colata imagens da webcan em loop - precisa interromper a seção para que a coleta de imagem seja interrompida

In [ ]:
# Instalar dependências
!pip install opencv-python

In [ ]:
import cv2
import mediapipe as mp
from google.colab.patches import cv2_imshow

import numpy as np
import base64
from PIL import Image
from io import BytesIO

# Ativar webcam no Google Colab
from google.colab import output
from IPython.display import Javascript

mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils

# Inicializa o módulo de detecção de mãos
hands = mp_hands.Hands(
    static_image_mode=False,
    max_num_hands=2,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5,
)

def show_video():
    js = Javascript("""
        var video = document.createElement('video');
        video.setAttribute('autoplay', '');
        video.setAttribute('playsinline', '');
        video.style.display = 'none';
        document.body.appendChild(video);

        navigator.mediaDevices.getUserMedia({ video: true })
            .then(stream => { video.srcObject = stream; });

        google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);

        async function captureFrame() {
            const canvas = document.createElement('canvas');
            canvas.width = video.videoWidth;
            canvas.height = video.videoHeight;
            const ctx = canvas.getContext('2d');
            ctx.drawImage(video, 0, 0);
            return canvas.toDataURL('image/jpeg', 0.8);
        }

        async function main() {
            while (true) {
                const img = await captureFrame();
                google.colab.kernel.invokeFunction(
                    'process_frame',
                    [img],
                    {}
                );
                await new Promise(r => setTimeout(r, 100));
            }
        }

        main();
    """)
    display(js)

# Função para processar cada frame

def process_frame(data):
    img_bytes = base64.b64decode(data.split(',')[1])
    img = Image.open(BytesIO(img_bytes))
    frame = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR)

    # Processa com MediaPipe
    results = hands.process(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))

    if results.multi_hand_landmarks:
        for hand_landmarks in results.multi_hand_landmarks:
            mp_drawing.draw_landmarks(
                frame,
                hand_landmarks,
                mp_hands.HAND_CONNECTIONS,
            )

    cv2_imshow(frame)

output.register_callback('process_frame', process_frame)

# Iniciar captura
show_video()
